In [7]:
%pip install sentencepiece --no-cache-dir

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import pandas as pd
import numpy as np
import requests
import torch
from tqdm import tqdm
import pickle
import os
import time

# =========================
# CONFIG
# =========================
PAIR_CSV = "./dataset/final_processed_data.csv"
COL1 = "SMILES1"
COL2 = "SMILES2"

TARGETS_CSV = "./dataset/bio-decagon-targets-all.csv"
OUTPUT_PATH = "drug_protein_embeddings.pkl"

device = torch.device("cpu")

MAX_PROTEINS_PER_DRUG = 3
SLEEP = 0.2

# =========================
# LOAD MODEL (FIXED TOKENIZER)
# =========================
from transformers import BertTokenizer, BertModel

tokenizer = BertTokenizer.from_pretrained("Rostlab/prot_bert", do_lower_case=False)
model = BertModel.from_pretrained("Rostlab/prot_bert")
model = model.to(device)
model.eval()

# =========================
# API FUNCTIONS
# =========================
def get_cid(smiles):
    try:
        url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/smiles/{smiles}/cids/JSON"
        r = requests.get(url, timeout=10)
        if r.status_code != 200:
            return None
        cid = r.json()['IdentifierList']['CID'][0]
        return f"CID{cid:09d}"
    except:
        return None


def gene_to_uniprot(gene_id):
    try:
        url = f"https://rest.uniprot.org/uniprotkb/search?query=gene_exact:{gene_id}+AND+organism_id:9606&format=json&size=1"
        r = requests.get(url, timeout=10)
        if r.status_code != 200:
            return None
        data = r.json()
        results = data.get("results", [])
        if not results:
            return None
        return results[0]["primaryAccession"]
    except:
        return None


def uniprot_to_sequence(acc):
    try:
        url = f"https://rest.uniprot.org/uniprotkb/{acc}.fasta"
        r = requests.get(url, timeout=10)
        if r.status_code != 200:
            return None
        fasta = r.text
        return "".join(fasta.split("\n")[1:])
    except:
        return None


# =========================
# ENCODER
# =========================
def encode_protein(seq):
    seq = " ".join(list(seq))

    inputs = tokenizer(
        seq,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    return outputs.last_hidden_state.mean(dim=1).cpu().numpy().flatten()


# =========================
# LOAD DATA
# =========================
pairs = pd.read_csv(PAIR_CSV)
targets = pd.read_csv(TARGETS_CSV)

cid_to_genes = targets.groupby("STITCH")["Gene"].apply(list).to_dict()

all_smiles = set(pairs[COL1]).union(set(pairs[COL2]))
print(f"Unique drugs: {len(all_smiles)}")

# =========================
# CACHE LOAD
# =========================
if os.path.exists(OUTPUT_PATH):
    with open(OUTPUT_PATH, "rb") as f:
        protein_embeddings = pickle.load(f)
else:
    protein_embeddings = {}

# =========================
# STEP 1: SMILES → CID (UNIQUE)
# =========================
print("Mapping SMILES → CID...")
smiles_to_cid = {}

for sm in tqdm(all_smiles):
    smiles_to_cid[sm] = get_cid(sm)
    time.sleep(SLEEP)

# =========================
# STEP 2: UNIQUE GENE IDs
# =========================
all_gene_ids = set(targets["Gene"].unique())
print(f"Unique genes: {len(all_gene_ids)}")

# =========================
# STEP 3: GENE → UNIPROT (ONCE)
# =========================
print("Mapping Gene → UniProt...")
gene_to_uniprot_cache = {}

for gid in tqdm(all_gene_ids):
    gene_to_uniprot_cache[gid] = gene_to_uniprot(gid)
    time.sleep(SLEEP)

# =========================
# STEP 4: UNIQUE UNIPROT IDS
# =========================
all_uniprot_ids = set(
    acc for acc in gene_to_uniprot_cache.values() if acc is not None
)

print(f"Unique UniProt IDs: {len(all_uniprot_ids)}")

# =========================
# STEP 5: FETCH SEQUENCES ONCE
# =========================
print("Fetching sequences...")
uniprot_seq_cache = {}

for acc in tqdm(all_uniprot_ids):
    uniprot_seq_cache[acc] = uniprot_to_sequence(acc)
    time.sleep(SLEEP)

# =========================
# STEP 6: ENCODE PROTEINS (CACHE)
# =========================
print("Encoding proteins...")
protein_seq_embedding_cache = {}

for acc, seq in tqdm(uniprot_seq_cache.items()):
    if seq is None or len(seq) < 10:
        continue
    try:
        protein_seq_embedding_cache[acc] = encode_protein(seq)
    except:
        continue

# =========================
# STEP 7: BUILD DRUG EMBEDDINGS
# =========================
print("Building drug embeddings...")

for sm in tqdm(all_smiles):

    if sm in protein_embeddings:
        continue

    cid = smiles_to_cid.get(sm)

    if cid is None or cid not in cid_to_genes:
        protein_embeddings[sm] = np.zeros(1024)
        continue

    gene_ids = cid_to_genes[cid][:MAX_PROTEINS_PER_DRUG]

    vecs = []

    for gid in gene_ids:
        acc = gene_to_uniprot_cache.get(gid)
        if acc is None:
            continue

        emb = protein_seq_embedding_cache.get(acc)
        if emb is not None:
            vecs.append(emb)

    if len(vecs) == 0:
        protein_embeddings[sm] = np.zeros(1024)
    else:
        protein_embeddings[sm] = np.mean(np.array(vecs), axis=0)

# =========================
# SAVE
# =========================
with open(OUTPUT_PATH, "wb") as f:
    pickle.dump(protein_embeddings, f)

print("Saved drug_protein_embeddings.pkl")

Loading weights: 100%|██████████| 487/487 [00:00<00:00, 20703.69it/s]
BertModel LOAD REPORT from: Rostlab/prot_bert
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Unique drugs: 644
Mapping SMILES → CID...


100%|██████████| 644/644 [17:03<00:00,  1.59s/it] 


Unique genes: 7795
Mapping Gene → UniProt...


100%|██████████| 7795/7795 [36:56<00:00,  3.52it/s]  


...

Encoding proteins...


100%|██████████| 4/4 [00:00<00:00, 4002.20it/s]


Building drug embeddings...


100%|██████████| 644/644 [00:00<00:00, 39224.72it/s]

Saved drug_protein_embeddings.pkl


In [5]:
with open("drug_protein_embeddings.pkl", "rb") as f:
    emb = pickle.load(f)

sample_key = next(iter(emb))
print("Sample SMILES:", sample_key)
print("Shape:", emb[sample_key].shape)

Sample SMILES: C1=CC(=CC=C1C#N)C(C2=CC=C(C=C2)C#N)N3C=NC=N3
Shape: (1024,)
